# Exp9.1 — A2 Optimization Stability and Representation Failure Decomposition

Aggregation-only notebook for protocol `a2_stability_decomposition_v1`.

In [ ]:
from pathlib import Path
import json
import pandas as pd

def find_repo_root(start=Path.cwd()):
    for p in (start, *start.parents):
        if (p / 'scripts').exists() and (p / 'notebooks').exists():
            return p
    raise RuntimeError('Repository root not found')

repo = find_repo_root()
root = repo / 'notebooks' / 'artifacts' / 'experiment_9_1_a2_stability_decomposition' / 'a2_stability_decomposition_v1'
audit = json.loads((root / 'audit.json').read_text())
audit


## A. Old A2 reproduction

In [ ]:
repro = pd.read_csv(root / 'reproduction_runs.csv')
cols = [c for c in ['seed','reference_test_ba','test_balanced_accuracy','delta_vs_reference_test_ba','best_train_ba','collapsed'] if c in repro.columns]
display(repro[cols])


## B. Fold × seed factorial

Rows are data folds; columns are exact reusable model initializations.

In [ ]:
test_ba = pd.read_csv(root / 'test_ba_matrix.csv', index_col=0)
train_ba = pd.read_csv(root / 'best_train_ba_matrix.csv', index_col=0)
collapse = pd.read_csv(root / 'collapse_matrix.csv', index_col=0)
display(test_ba)
display(train_ba)
display(collapse)


## C. Fold effect vs seed effect

In [ ]:
fold_summary = pd.read_csv(root / 'fold_effect_summary.csv')
seed_summary = pd.read_csv(root / 'seed_effect_summary.csv')
collapse_summary = json.loads((root / 'collapse_summary.json').read_text())
display(fold_summary)
display(seed_summary)
collapse_summary


## D. Native A2 vs frozen L1/L2 probes

Use this table to distinguish optimization failure, L2 information loss and readout bottlenecks.

In [ ]:
probes = pd.read_csv(root / 'native_vs_probe.csv')
display(probes.sort_values(['collapsed','fold','seed']))


## E. Optimization trajectory diagnostics

In [ ]:
training = pd.read_csv(root / 'training_diagnostics.csv')
activity = pd.read_csv(root / 'activity_diagnostics.csv')
display(training.groupby(['fold','seed'])[['train_ba','val_ba','grad_l1','grad_l2','grad_out']].tail(1))
display(activity.head())


## Interpretation guide

- low train BA + low L1/L2 probes → backbone optimization failure;
- low native BA + high L2 probe → native objective/readout failure;
- high L1 probe + low L2 probe → L2 destroys useful representation;
- high train BA + low held-out BA → representation overfit;
- failures concentrated by seed → initialization sensitivity;
- failures concentrated by fold → subset difficulty;
- isolated fold×seed failures → nonlinear interaction.